In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from pathlib import Path
import os
import json

In [ ]:
projectRoot = Path.cwd().parents[2]

POIS_DENSITY = projectRoot / "data" / "clean" / "pois" / "zones_with_poi_group_density.parquet"
POIS_COUNT = projectRoot / "data" / "clean" / "pois" / "zones_with_poi_groups.parquet"
DEMAND = projectRoot / "data" / "clean" / "taxi_zone_demand.parquet"
HTML_DIR = projectRoot / "data" / "html" / "pois"
os.makedirs(HTML_DIR, exist_ok=True)

In [ ]:
POIS_COLUMNS = [
    'total_poi_density','food_drink_per_mi2','nightlife_alcohol_per_mi2','healthcare_per_mi2','accommodation_per_mi2',
    'education_per_mi2','entertainment_per_mi2', 'government_per_mi2',
    'commercial_per_mi2',
    'sports_fitness_per_mi2', 'tourism_per_mi2', 'transport_per_mi2']

DEMAND_COLUMNS = [   
    "avg_daily_pickups",
    "pickups_morning",
    "pickups_afternoon",
    "pickups_evening",
    "pickups_night"]

BOROUGH_COLORS = {
    "Manhattan": "#e74c3c",
    "Brooklyn": "#3498db",
    "Queens": "#2ecc71",
    "Bronx": "#f39c12",
    "Staten Island": "#9b59b6",
    "EWR": "#95a5a6",
}

LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, sans-serif", size=12),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
)

In [ ]:
DEMAND_LABELS = {
    'pickups_morning': 'Pickups Morning (6-12h)',
    'pickups_afternoon': 'Pickups Afternoon (12-18h)',
    'pickups_evening': 'Pickups Evening (18-24h)',
    'pickups_night': 'Pickups Night (0-6h)',
}

def clean_label(col: str) -> str:
    if col in DEMAND_LABELS:
        return DEMAND_LABELS[col]
    return col.replace('_per_mi2', '').replace('_', ' ').title()

# LOAD

In [ ]:
def load_and_merge(pois_path: str, demand_path: str) -> pd.DataFrame:
    
    pois = pd.read_parquet(pois_path)
    demand = pd.read_parquet(demand_path)

    # POIs usan 'id', demand usa 'PULocationID'
    pois = pois.rename(columns={"id": "PULocationID"})

    df = pois.merge(demand, on="PULocationID", how="inner")

    return df

In [ ]:
df = load_and_merge(POIS_DENSITY, DEMAND)
df["total_poi_density"] = df[POIS_COLUMNS[1:]].sum(axis=1)

In [ ]:
import geopandas as gpd
from shapely import wkt

gdf = df.copy()
gdf['geometry'] = gdf['geometry'].apply(wkt.loads) 
gdf = gpd.GeoDataFrame(gdf, crs='EPSG:4326')

gdf_proj = gdf.to_crs('EPSG:2263')
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['lat'] = gdf_proj.centroid.to_crs('EPSG:4326').y
gdf['lon'] = gdf_proj.centroid.to_crs('EPSG:4326').x



Load group counts dataset

In [ ]:
df_counts = pd.read_parquet(POIS_COUNT)
pois_columns_count = ['accommodation', 'commercial', 'education', 'entertainment',
       'food_drink', 'government', 'healthcare', 'nightlife_alcohol', 'other',
       'sports_fitness', 'tourism', 'transport']

In [ ]:
df_counts['total_count'] = df_counts[pois_columns_count].sum(axis=1)

In [ ]:
pois_columns_count.append('total_count')

In [ ]:
pois_columns_count

In [ ]:
df_counts[pois_columns_count].sum()

# Group Percentage

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df3 = pd.read_parquet(f"{projectRoot}/data/clean/pois/zones_with_poi_counts.parquet")

POI_GROUPS = {
        'transport': [
            'amenity_parking', 'amenity_taxi', 'amenity_fuel',
            'public_transport_station', 'toll_yes'
        ],
        'food_drink': [
            'amenity_restaurant', 'amenity_cafe', 'amenity_bar','amenity_fast_food'
        ],
        'healthcare': [
            'amenity_hospital', 'amenity_clinic', 'amenity_dentist'
        ],
        'entertainment': [
            'amenity_cinema', 'amenity_theatre', 'tourism_museum',
            'amenity_events_venue', 'amenity_music_venue'
        ],
        'tourism':[
            'tourism_attraction', 'tourism_theme_park', 'tourism_zoo',
            'tourism_yes', 'historic_building'
        ],
        'nightlife_alcohol': [
            'amenity_nightclub', 'amenity_pub', 'amenity_biergarten', 'amenity_casino',
            'amenity_stripclub'
        ],
        'accommodation': [
            'tourism_hotel', 'tourism_motel', 'building_hotel'
        ],
        'education': [
            'amenity_university', 'amenity_college', 'amenity_school', 'amenity_libray'
        ],
        'sports_fitness': [
            'leisure_fitness_centre', 'leisure_stadium'
        ],
        'government': [
            'amenity_courthouse', 'office_diplomatic'
        ],
        'commercial': [
            'shop_mall', 'building_office'
        ]
    }

# Nombres custom para POIs ambiguos
POI_LABELS = {
    'tourism_hotel': 'Hotel (Tourism)',
    'building_hotel': 'Hotel (Building)',
    'toll_yes': 'Toll/Peaje',
    'tourism_yes': 'Atracción General',
}

MIN_PCT = 3  # % umbral: por debajo se agrupa en "Otros"

def poi_label(poi_key: str) -> str:
    if poi_key in POI_LABELS:
        return POI_LABELS[poi_key]
    return poi_key.split('_', 1)[1].replace('_', ' ').title()

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
          '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']
otros_color = '#AAAAAA'

# Preparar datos agrupando los pequeños en "Otros"
group_data = {}
for group, pois in POI_GROUPS.items():
    existing = [p for p in pois if p in df3.columns]
    totals = df3[existing].sum().sort_values(ascending=False)
    total = totals.sum()
    if total == 0:
        continue

    pcts = (totals / total * 100).round(1)
    labels, values, final_pcts = [], [], []
    otros_val, otros_pct = 0, 0.0

    for poi, val in totals.items():
        pct = round(val / total * 100, 1)
        if pct >= MIN_PCT:
            labels.append(poi_label(poi))
            values.append(val)
            final_pcts.append(pct)
        else:
            otros_val += val
            otros_pct += pct

    if otros_val > 0:
        labels.append('Otros')
        values.append(otros_val)
        final_pcts.append(round(otros_pct, 1))

    group_data[group] = {
        'labels': labels, 'values': values,
        'pcts': final_pcts, 'total': total
    }

group_data = dict(sorted(group_data.items(), key=lambda x: x[1]['total'], reverse=True))

# Construir figura
fig = go.Figure()

for group, data in group_data.items():
    group_label = group.replace('_', ' ').title()

    for i, (label, val, pct) in enumerate(zip(data['labels'], data['values'], data['pcts'])):
        color = otros_color if label == 'Otros' else colors[i % len(colors)]

        fig.add_trace(go.Bar(
            y=[group_label], x=[pct],
            orientation='h', showlegend=False,
            marker_color=color,
            marker_line=dict(width=0.5, color='black'),
            text=f'{label}: {pct}%',
            textposition='inside',
            textfont=dict(size=16, color='black'),
            hovertemplate=f'{label}<br>{val:,} POIs ({pct}%)<extra>{group_label}</extra>',
        ))

# Totales a la derecha
for group, data in group_data.items():
    fig.add_annotation(
        y=group.replace('_', ' ').title(), x=100,
        text=f'  <b>{data["total"]:,}</b>',
        showarrow=False, xanchor='left',
        font=dict(size=15, color='#333'),
    )

fig.update_layout(
    title=dict(text='<b>Composición Porcentual de POIs por Grupo<b>',
               font=dict(size=20), x=0.5),
    xaxis=dict(title='Porcentaje (%)', range=[0, 112],
               ticksuffix='%', tickfont=dict(size=16),
               gridcolor='rgba(0,0,0,0.05)', showline=False),
    yaxis=dict(tickfont=dict(size=14), autorange='reversed'),
    barmode='stack',
    height=550, width=1200,
    margin=dict(t=80, b=60, l=140, r=80),
    hoverlabel=dict(bgcolor='white', font_size=12),
    template = 'plotly_white'
)
fig.write_html(HTML_DIR / f"pois_group_composition.html")
fig.show()

# Counts

In [ ]:
import plotly.graph_objects as go

categories = pois_columns_count
cat_totals = df_counts[categories].sum().sort_values(ascending=False)
sorted_cats = cat_totals.index.tolist()
sorted_display = [c.replace('_', ' ').title() for c in sorted_cats]

borough_totals = df_counts.groupby('borough')[categories].sum()
borough_totals = borough_totals.loc[borough_totals.sum(axis=1).sort_values(ascending=False).index]

fig = go.Figure()

for borough in borough_totals.index:
    vals = borough_totals.loc[borough, sorted_cats].values
    fig.add_trace(go.Bar(
        name=borough,
        x=sorted_display,
        y=vals,
        marker_color=BOROUGH_COLORS.get(borough, '#636EFA'),
        marker_line=dict(width=0.5, color='rgba(0,0,0,0.15)'),
        text=[f'{v:,.0f}' for v in vals],
        textposition='inside',
        textfont=dict(size=10, color='white'),
        hovertemplate=f'<b>{borough}</b><br>Categoría: %{{x}}<br>POIs: %{{y:,.0f}}<extra></extra>',
    ))

# Totales encima de cada barra apilada
max_y = cat_totals.max()
for display, key in zip(sorted_display, sorted_cats):
    fig.add_annotation(
        x=display, y=borough_totals[key].sum(),
        text=f'<b>{borough_totals[key].sum():,.0f}</b>',
        showarrow=False, yshift=12,
        font=dict(size=11, color='#1a1a2e'),
    )

fig.update_layout(
    title=dict(text='Distribución de Puntos de Interés por Categoría y Borough',
               font=dict(size=22, color='#1a1a2e'), x=0.5),
    xaxis=dict(title='Categoría de POI', tickangle=-35, tickfont=dict(size=12),
               categoryorder='array', categoryarray=sorted_display,
               showline=True, linewidth=1, linecolor='rgba(0,0,0,0.1)'),
    yaxis=dict(title='Cantidad de POIs', tickfont=dict(size=12),
               gridcolor='rgba(0,0,0,0.06)', separatethousands=True,
               range=[0, max_y * 1.08], showline=False),
    barmode='stack',
    legend=dict(title='Borough', font=dict(size=12), orientation='h',
                yanchor='bottom', y=1.02, xanchor='center', x=0.5,
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor='rgba(0,0,0,0.1)', borderwidth=1),
    plot_bgcolor='white', paper_bgcolor='#fafafa',
    height=650, width=1200,
    margin=dict(t=100, b=100, l=80, r=40),
    hoverlabel=dict(bgcolor='white', font_size=13),
)
fig.write_html(HTML_DIR / f"pois_distribution_borough.html")
fig.show()

# Density

## POIs density vs Demand

In [ ]:
from scipy import stats
def total_density_vs_pickups(df: pd.DataFrame, poi_cols: list[str]) -> go.Figure:
    """
    Scatter plot of Total POIs density vs daily pickups

    r = Pearson's correlation [-1, 1]
    p = p-value
    """
    df = df.copy()
    df["total_poi_density"] = df[poi_cols].sum(axis=1)

    r, p = stats.pearsonr(df["total_poi_density"], df["avg_daily_pickups"])

    fig = px.scatter(
        df, x="total_poi_density", y="avg_daily_pickups",
        color="borough",
        hover_name="zone" if "zone" in df.columns else None,
        color_discrete_map=BOROUGH_COLORS,
        labels={"total_poi_density": "Densidad total POIs (POIs/mi²)",
                "avg_daily_pickups": "Media pickups diarios"},
        title=f"<b>Densidad total de POIs vs Demanda</b>"
              f"<br><sup>Pearson r = {r:.3f} (p = {p:.2e})</sup>",
        opacity=0.7,
    )

    coef = np.polyfit(df["total_poi_density"], df["avg_daily_pickups"], 1)
    x_line = np.linspace(df["total_poi_density"].min(), df["total_poi_density"].max(), 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(coef, x_line),
        mode="lines", name=f"Tendencia (r={r:.3f})",
        line=dict(color="black", dash="dash", width=2),
    ))
    fig.update_layout(height=550,width=1000, template="plotly_white")
    fig.write_html(HTML_DIR / f"pois_density_demand.html")
    return fig

In [ ]:
total_density_vs_pickups(df, POIS_COLUMNS)

## CORRELATION HEATMAP

In [ ]:
def heatmap(df: pd.DataFrame, poi_cols: list[str], demand_cols: list[str]) -> go.Figure:

    extra = [c for c in ["avg_tip_pct", "avg_fare", "avg_tip", "trips_yellow", "trips_fhv","trips_fhvhv"] if c in df.columns]
    all_demand = demand_cols + extra

    sub = df[poi_cols + all_demand].corr().loc[poi_cols, all_demand]

    fig = go.Figure(data=go.Heatmap(
        z=sub.values,
        x=[clean_label(c) for c in all_demand],
        y=[clean_label(c) for c in poi_cols],
        colorscale="RdYlGn", zmin=-0.3, zmax=1.0,
        text=np.round(sub.values, 2), texttemplate="%{text}",
        textfont={"size": 11},
        colorbar=dict(title="r"),
    ))
    fig.update_layout(
        title="<b>Correlation - POI Groups x Demand</b>"
              "<br><sup>Green = High Correlation</sup>",
        height=500, width=max(750, 110 * len(all_demand)),
        template="plotly_white",
        yaxis=dict(autorange="reversed"),
    )
    fig.write_html(HTML_DIR / f"pois_heatmap_demand.html")
    return fig

In [ ]:
heatmap(df, POIS_COLUMNS[1:], DEMAND_COLUMNS)

## MAPS

In [ ]:
def pois_density_map(gdf: gpd.GeoDataFrame,poi_col:str):

    """
    Plots a choropleth_map of each zone with density of POIs
    """

    name = clean_label(poi_col)

    if not "Density" in name:
        name + " Density"

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df,
        geojson=geojson,
        locations="zone",
        featureidkey="properties.zone",
        color=poi_col,
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        #range_color=[0, 100_000],
        opacity=0.7,
        title=f"<b>{name} per Zone<b>",
        labels={poi_col: f"<b>{name}<b><br>(per mi²)"},
        
    )

    fig.update_layout(
        margin={"r": 0, "t": 75, "l": 50, "b": 50},
        width=1200,
        height=800,
        font=dict(size=16),  # Tamaño general de fuente
        title_font=dict(size=24), # Tamaño del título
        coloraxis_colorbar=dict(
            title_font=dict(size=18),  # Tamaño del título de la leyenda
            tickfont=dict(size=14)  # Tamaño de los números de la leyenda
        )
    )

    fig.show()
    fig.write_html(HTML_DIR / f"pois_{poi_col}_map.html")

In [ ]:
pois_density_map(gdf, 'total_poi_density')

    Nota: Si al cambiar de selección en el dropdown, los perimetros
    de los Top y Bottom se ven tenues, hay que seleccionar de nuevo 
    el grupo de POI en el dropdown.
    

In [ ]:
def pois_density_map_top_bottom(gdf: gpd.GeoDataFrame, poi_columns: list):
    """
    Mapa choropleth interactivo con dropdown para seleccionar POI.
    Resalta Top 5 (borde verde) y Bottom 5 (borde rojo) con ranking en texto.

    Nota: Si al cambiar de selección en el dropdown, los perimetros
    de los Top y Bottom se ven tenues, hay que seleccionar de nuevo 
    el grupo de POI en el dropdown.
    """
    
    gdf_map = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_map.to_json())
    
    fig = go.Figure()
    
    trace_ranges = []
    annotations_list = []
    
    def get_polygon_coords(feature):
        """Devuelve lista de (lons, lats) para cada anillo exterior."""
        coords = feature['geometry']['coordinates']
        geom_type = feature['geometry']['type']
        polygons = coords if geom_type == 'MultiPolygon' else [coords]
        result = []
        for poly in polygons:
            lons = [c[0] for c in poly[0]]
            lats = [c[1] for c in poly[0]]
            result.append((lons, lats))
        return result
    
    def collect_border_coords(zone_list):
        """Recopila coordenadas de todas las zonas en una sola lista con separadores None."""
        all_lons = []
        all_lats = []
        for feature in geojson['features']:
            if feature['properties']['zone'] not in zone_list:
                continue
            for lons, lats in get_polygon_coords(feature):
                all_lons.extend(lons + [None])  # None separa polígonos
                all_lats.extend(lats + [None])
        return all_lons, all_lats
    
    for poi_type in poi_columns:
        poi_name = poi_type.replace('_per_mi2', '').replace('_', ' ').title()
        start_idx = len(fig.data)
        
        # --- Top 5 / Bottom 5 ---
        df_non_zero = gdf_map[gdf_map[poi_type] > 0]
        top_5_zones = gdf_map.nlargest(5, poi_type)['zone'].tolist()
        bottom_5_zones = (
            df_non_zero.nsmallest(5, poi_type)['zone'].tolist()
            if len(df_non_zero) >= 5
            else gdf_map.nsmallest(5, poi_type)['zone'].tolist()
        )
        
        # --- Texto de ranking ---
        top_5_data = gdf_map.nlargest(5, poi_type)[['zone', poi_type]]
        bottom_5_data = (
            df_non_zero.nsmallest(5, poi_type)[['zone', poi_type]]
            if len(df_non_zero) >= 5
            else gdf_map.nsmallest(5, poi_type)[['zone', poi_type]]
        )
        
        ranking_text = "<b>Top 5 Zonas:</b><br>"
        for i, (_, row) in enumerate(top_5_data.iterrows(), 1):
            name = row['zone'][:22]
            ranking_text += f"{i}. {name}: {row[poi_type]:.1f}<br>"
        ranking_text += "<br><b>Bottom 5 Zonas:</b><br>"
        for i, (_, row) in enumerate(bottom_5_data.iterrows(), 1):
            name = row['zone'][:22]
            ranking_text += f"{i}. {name}: {row[poi_type]:.1f}<br>"
        
        # --- 1) Choropleth con borde gris fino en todas las zonas ---
        fig.add_trace(go.Choroplethmap(
            geojson=geojson,
            locations=gdf_map['zone'],
            z=gdf_map[poi_type],
            featureidkey="properties.zone",
            colorscale="YlOrRd",
            marker_opacity=0.7,
            marker_line_width=1,
            marker_line_color="gray",
            colorbar=dict(
                title=f"<b>{poi_name}</b><br>(per mi²)",
                title_font=dict(size=16),
                tickfont=dict(size=14),
                len=0.40,
                thickness=20,
                x=1.02,          # fuera del mapa a la derecha
                xanchor="left",
                y=0.95,          # arriba, encima de las anotaciones
                yanchor="top",
            ),
            hovertemplate=(
                '<b>%{location}</b><br>'
                f'{poi_name}: ' + '%{z:.2f} POIs/mi²<extra></extra>'
            ),
            visible=False,
            name=poi_name
        ))
        
        # --- 2) UNA sola traza verde para Top 5 ---
        top_lons, top_lats = collect_border_coords(top_5_zones)
        if top_lons:
            fig.add_trace(go.Scattermap(
                lon=top_lons,
                lat=top_lats,
                mode='lines',
                line=dict(width=3.0, color='green'),
                showlegend=False,
                hoverinfo='skip',
                visible=False,
                name=f'{poi_name}_top5'
            ))
        
        # --- 3) UNA sola traza roja para Bottom 5 ---
        bot_lons, bot_lats = collect_border_coords(bottom_5_zones)
        if bot_lons:
            fig.add_trace(go.Scattermap(
                lon=bot_lons,
                lat=bot_lats,
                mode='lines',
                line=dict(width=3.0, color='red'),
                showlegend=False,
                hoverinfo='skip',
                visible=False,
                name=f'{poi_name}_bot5'
            ))
        
        end_idx = len(fig.data)
        trace_ranges.append((start_idx, end_idx))
        
        # --- Anotación debajo de la colorbar ---
        annotations_list.append(dict(
            text=ranking_text,
            xref="paper", yref="paper",
            x=1.02, y=0.52,         # debajo de la colorbar
            xanchor="left", yanchor="top",
            showarrow=False,
            font=dict(size=14, family="monospace"),
            align="left",
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="black",
            borderwidth=1,
            borderpad=10,
            visible=False
        ))
    
    # --- Hacer visible la primera categoría ---
    s, e = trace_ranges[0]
    for i in range(s, e):
        fig.data[i].visible = True
    annotations_list[0]['visible'] = True
    
    # --- Dropdown buttons ---
    total_traces = len(fig.data)
    buttons = []
    
    for poi_idx, poi_type in enumerate(poi_columns):
        poi_name = poi_type.replace('_per_mi2', '').replace('_', ' ').title()
        
        visible = [False] * total_traces
        s, e = trace_ranges[poi_idx]
        for i in range(s, e):
            visible[i] = True
        
        anns = []
        for ann_i, ann in enumerate(annotations_list):
            ann_copy = dict(ann)
            ann_copy['visible'] = (ann_i == poi_idx)
            anns.append(ann_copy)
        
        buttons.append(dict(
            label=poi_name,
            method='update',
            args=[
                {'visible': visible},
                {
                    'title': (
                        f'<b>{poi_name} - Densidad por Zona</b>'
                        f'<br><sub>Borde Verde: Top 5 | Borde Rojo: Bottom 5</sub>'
                    ),
                    'annotations': anns
                }
            ]
        ))
    
    # --- Layout ---
    first_name = poi_columns[0].replace('_per_mi2', '').replace('_', ' ').title()
    fig.update_layout(
        map=dict(
            style="carto-positron",
            zoom=9.5,
            center={"lat": 40.7128, "lon": -74.0060}
        ),
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.01, xanchor="left",
            y=0.99, yanchor="top",
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1
        )],
        title=(
            f'<b>{first_name} - Densidad por Zona</b>'
            f'<br><sub>Borde Verde: Top 5 | Borde Rojo: Bottom 5</sub>'
        ),
        annotations=annotations_list,
        margin={"r": 350, "t": 100, "l": 50, "b": 50},
        width=1500,
        height=800,
        font=dict(size=14),
        title_font=dict(size=20)
    )
    
    fig.show()
    fig.write_html(HTML_DIR / "pois_top5_bottom5_map.html")
    return fig

fig = pois_density_map_top_bottom(gdf, POIS_COLUMNS)